<a href="https://colab.research.google.com/github/giorgioganemfederle-dotcom/Cuadra-Zona-Centro/blob/main/Cuadra_Norte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy requests folium ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 21.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


In [ ]:
import pandas as pd
import numpy as np
import requests
import folium

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

print("Programa iniciado correctamente")

Programa iniciado correctamente


In [ ]:
datos = [

    # Centro de distribución
    ["León", "Guanajuato", "Centro", 21.1220, -101.6840],

    # Baja California
    ["Tijuana", "Baja California", "Norte", 32.5149, -117.0382],

    # Baja California Sur
    ["Cabo San Lucas", "Baja California Sur", "Norte", 22.8905, -109.9167],
    ["San José del Cabo", "Baja California Sur", "Norte", 23.0580, -109.6977],

    # Sonora
    ["Hermosillo", "Sonora", "Norte", 29.0729, -110.9559],
    ["Ciudad Obregón", "Sonora", "Norte", 27.4864, -109.9408],

    # Chihuahua
    ["Chihuahua", "Chihuahua", "Norte", 28.6320, -106.0691],

    # Coahuila
    ["Saltillo", "Coahuila", "Norte", 25.4383, -100.9737],
    ["Torreón", "Coahuila", "Norte", 25.5428, -103.4068],
    ["Cuatro Ciénegas", "Coahuila", "Norte", 26.9868, -102.0628],

    # Nuevo León
    ["Monterrey", "Nuevo León", "Norte", 25.6866, -100.3161],
    ["San Pedro Garza García", "Nuevo León", "Norte", 25.6572, -100.4020],
    ["San Nicolás de los Garza", "Nuevo León", "Norte", 25.7500, -100.3000],
    ["Escobedo", "Nuevo León", "Norte", 25.7900, -100.3300],

    # Tamaulipas
    ["Reynosa", "Tamaulipas", "Norte", 26.0922, -98.2778],
    ["Tampico", "Tamaulipas", "Norte", 22.2331, -97.8611],

    # Sinaloa
    ["Culiacán", "Sinaloa", "Norte", 24.8091, -107.3940],
    ["Mazatlán", "Sinaloa", "Norte", 23.2494, -106.4111],
    ["Los Mochis", "Sinaloa", "Norte", 25.7900, -108.9850],

    # Durango
    ["Durango", "Durango", "Norte", 24.0277, -104.6532],

    # Zacatecas
    ["Zacatecas", "Zacatecas", "Norte", 22.7709, -102.5832],

    # San Luis Potosí
    ["San Luis Potosí", "San Luis Potosí", "Norte", 22.1565, -100.9855],

    # Nayarit
    ["Ixtlán del Río", "Nayarit", "Norte", 21.0350, -104.3710],
    ["Xalisco", "Nayarit", "Norte", 21.4470, -104.9000]
]

df = pd.DataFrame(
    datos,
    columns=[
        "Ciudad",
        "Estado",
        "Region",
        "Latitud",
        "Longitud"
    ]
)

df

,Ciudad,Estado,Region,Latitud,Longitud
0,León,Guanajuato,Centro,21.1220,-101.6840
1,Tijuana,Baja California,Norte,32.5149,-117.0382
2,Cabo San Lucas,Baja California Sur,Norte,22.8905,-109.9167
3,San José del Cabo,Baja California Sur,Norte,23.0580,-109.6977
4,Hermosillo,Sonora,Norte,29.0729,-110.9559
5,Ciudad Obregón,Sonora,Norte,27.4864,-109.9408
6,Chihuahua,Chihuahua,Norte,28.6320,-106.0691
7,Saltillo,Coahuila,Norte,25.4383,-100.9737
8,Torreón,Coahuila,Norte,25.5428,-103.4068
9,Cuatro Ciénegas,Coahuila,Norte,26.9868,-102.0628


In [ ]:
print("Número de puntos:", len(df))
print("Centro de distribución: León")
print("Destinos:", len(df) - 1)

Número de puntos: 24
Centro de distribución: León
Destinos: 23


In [ ]:
mapa = folium.Map(
    location=[24, -105],
    zoom_start=5
)

for _, fila in df.iterrows():

    if fila["Ciudad"] == "León":
        color = "red"
    else:
        color = "blue"

    folium.Marker(
        location=[
            fila["Latitud"],
            fila["Longitud"]
        ],
        popup=(
            f"{fila['Ciudad']}<br>"
            f"{fila['Estado']}<br>"
            f"{fila['Region']}"
        ),
        tooltip=fila["Ciudad"],
        icon=folium.Icon(
            color=color
        )
    ).add_to(mapa)

mapa

In [ ]:
def obtener_ruta(origen, destino):

    lon1 = origen[1]
    lat1 = origen[0]

    lon2 = destino[1]
    lat2 = destino[0]

    url = (
        f"https://router.project-osrm.org/"
        f"route/v1/driving/"
        f"{lon1},{lat1};{lon2},{lat2}"
        f"?overview=false"
    )

    respuesta = requests.get(url)

    datos_ruta = respuesta.json()

    if datos_ruta["code"] == "Ok":

        ruta = datos_ruta["routes"][0]

        distancia = ruta["distance"] / 1000
        tiempo = ruta["duration"] / 60

        return distancia, tiempo

    else:

        return None, None

In [ ]:
leon = (
    df.loc[df["Ciudad"] == "León", "Latitud"].iloc[0],
    df.loc[df["Ciudad"] == "León", "Longitud"].iloc[0]
)

monterrey = (
    df.loc[df["Ciudad"] == "Monterrey", "Latitud"].iloc[0],
    df.loc[df["Ciudad"] == "Monterrey", "Longitud"].iloc[0]
)

distancia, tiempo = obtener_ruta(
    leon,
    monterrey
)

print(f"León → Monterrey")
print(f"Distancia: {distancia:.2f} km")
print(f"Tiempo: {tiempo/60:.2f} horas")

León → Monterrey
Distancia: 689.95 km
Tiempo: 8.83 horas


In [ ]:
n = len(df)

matriz_distancia = np.zeros((n, n))
matriz_tiempo = np.zeros((n, n))

for i in range(n):

    for j in range(n):

        if i == j:
            continue

        origen = (
            df.iloc[i]["Latitud"],
            df.iloc[i]["Longitud"]
        )

        destino = (
            df.iloc[j]["Latitud"],
            df.iloc[j]["Longitud"]
        )

        distancia, tiempo = obtener_ruta(
            origen,
            destino
        )

        matriz_distancia[i][j] = distancia
        matriz_tiempo[i][j] = tiempo

print("Matrices creadas.")

Matrices creadas.


In [ ]:
distancia_normalizada = (
    matriz_distancia / matriz_distancia.max()
)

tiempo_normalizado = (
    matriz_tiempo / matriz_tiempo.max()
)

In [ ]:
peso_distancia = 0.5
peso_tiempo = 0.5

matriz_costo = (
    peso_distancia * distancia_normalizada
    +
    peso_tiempo * tiempo_normalizado
)

In [ ]:
def optimizar_ruta(matriz_costo):

    n = len(matriz_costo)

    # León = índice 0
    manager = pywrapcp.RoutingIndexManager(
        n,
        1,
        0
    )

    routing = pywrapcp.RoutingModel(
        manager
    )

    def costo_callback(
        from_index,
        to_index
    ):

        origen = manager.IndexToNode(
            from_index
        )

        destino = manager.IndexToNode(
            to_index
        )

        return int(
            matriz_costo[origen][destino] * 1000000
        )

    callback_index = (
        routing.RegisterTransitCallback(
            costo_callback
        )
    )

    routing.SetArcCostEvaluatorOfAllVehicles(
        callback_index
    )

    parametros = (
        pywrapcp.DefaultRoutingSearchParameters()
    )

    parametros.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy
        .PATH_CHEAPEST_ARC
    )

    parametros.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic
        .GUIDED_LOCAL_SEARCH
    )

    parametros.time_limit.seconds = 60

    solucion = routing.SolveWithParameters(
        parametros
    )

    if solucion is None:

        return None

    ruta = []

    index = routing.Start(0)

    while not routing.IsEnd(index):

        nodo = manager.IndexToNode(index)

        ruta.append(nodo)

        index = solucion.Value(
            routing.NextVar(index)
        )

    ruta.append(
        manager.IndexToNode(index)
    )

    return ruta

In [ ]:
ruta_optima = optimizar_ruta(
    matriz_costo
)

In [ ]:
print("RUTA OPTIMIZADA")
print("================")

for i, nodo in enumerate(ruta_optima):

    print(
        f"{i+1}. "
        f"{df.iloc[nodo]['Ciudad']}"
    )

RUTA OPTIMIZADA
1. León
2. Ixtlán del Río
3. Xalisco
4. Mazatlán
5. Culiacán
6. Los Mochis
7. Ciudad Obregón
8. Hermosillo
9. San José del Cabo
10. Cabo San Lucas
11. Tijuana
12. Chihuahua
13. Durango
14. Torreón
15. Cuatro Ciénegas
16. Saltillo
17. San Pedro Garza García
18. Monterrey
19. Escobedo
20. San Nicolás de los Garza
21. Reynosa
22. Tampico
23. San Luis Potosí
24. Zacatecas
25. León


In [ ]:
distancia_total = 0
tiempo_total = 0

for i in range(len(ruta_optima) - 1):

    origen = ruta_optima[i]
    destino = ruta_optima[i + 1]

    distancia_total += (
        matriz_distancia[origen][destino]
    )

    tiempo_total += (
        matriz_tiempo[origen][destino]
    )

print(
    f"Distancia total: "
    f"{distancia_total:,.2f} km"
)

print(
    f"Tiempo total: "
    f"{tiempo_total/60:,.2f} horas"
)

Distancia total: 9,953.17 km
Tiempo total: 133.74 horas


In [ ]:
mapa_ruta = folium.Map(
    location=[24, -105],
    zoom_start=5
)

for numero, nodo in enumerate(ruta_optima):

    fila = df.iloc[nodo]

    if nodo == 0:

        color = "red"

    else:

        color = "blue"

    folium.Marker(
        location=[
            fila["Latitud"],
            fila["Longitud"]
        ],
        popup=(
            f"Parada {numero + 1}<br>"
            f"{fila['Ciudad']}<br>"
            f"{fila['Estado']}"
        ),
        tooltip=(
            f"{numero + 1}. "
            f"{fila['Ciudad']}"
        ),
        icon=folium.Icon(
            color=color
        )
    ).add_to(mapa_ruta)

mapa_ruta

In [ ]:
def obtener_geometria_ruta(ruta):

    puntos = []

    for nodo in ruta:

        fila = df.iloc[nodo]

        puntos.append(
            f"{fila['Longitud']},"
            f"{fila['Latitud']}"
        )

    coordenadas = ";".join(puntos)

    url = (
        "https://router.project-osrm.org/"
        "route/v1/driving/"
        f"{coordenadas}"
        "?overview=full"
        "&geometries=geojson"
    )

    respuesta = requests.get(url)

    return respuesta.json()

In [ ]:
datos_gps = obtener_geometria_ruta(
    ruta_optima
)

In [ ]:
geometria = (
    datos_gps["routes"][0]["geometry"]
)

folium.GeoJson(
    geometria,
    name="Ruta optimizada Cuadra"
).add_to(mapa_ruta)

folium.LayerControl().add_to(
    mapa_ruta
)

mapa_ruta

In [ ]:
ruta_1 = [
    "León",
    "Ixtlán del Río",
    "Xalisco",
    "Mazatlán",
    "Culiacán",
    "Los Mochis",
    "Ciudad Obregón",
    "Hermosillo",
    "Tijuana",


]

print("Ruta 1 - PACÍFICO NORTE")
print("=" * 30)

for i, ciudad in enumerate(ruta_1, start=1):
  print(f"{i}. {ciudad}")

Ruta 1 - PACÍFICO NORTE
1. León
2. Ixtlán del Río
3. Xalisco
4. Mazatlán
5. Culiacán
6. Los Mochis
7. Ciudad Obregón
8. Hermosillo
9. Tijuana


In [ ]:
coordenadas_ruta_1 = {

    "León": (21.1220, -101.6840),

    "Ixtlán del Río": (21.0350, -104.3710),

    "Xalisco": (21.4470, -104.9000),

    "Mazatlán": (23.2494, -106.4111),

    "Culiacán": (24.8091, -107.3940),

    "Los Mochis": (25.7900, -108.9850),

    "Ciudad Obregón": (27.4864, -109.9408),

    "Hermosillo": (29.0729, -110.9559),

    "Tijuana": (32.5149, -117.0382)
}

In [ ]:
import folium

mapa_ruta_1 = folium.Map(
    location=[25, -107],
    zoom_start=5
)

for numero, ciudad in enumerate(ruta_1, start=1):

    latitud, longitud = coordenadas_ruta_1[ciudad]

    color = "red" if ciudad == "León" else "blue"

    folium.Marker(
        location=[latitud, longitud],
        popup=f"{numero}. {ciudad}",
        tooltip=f"{numero}. {ciudad}",
        icon=folium.Icon(color=color)
    ).add_to(mapa_ruta_1)

mapa_ruta_1

In [ ]:
import requests

def obtener_ruta_carretera(ciudades, coordenadas):

    puntos = []

    for ciudad in ciudades:

        latitud, longitud = coordenadas[ciudad]

        puntos.append(
            f"{longitud},{latitud}"
        )

    cadena = ";".join(puntos)

    url = (
        "https://router.project-osrm.org/"
        "route/v1/driving/"
        + cadena +
        "?overview=full"
        "&geometries=geojson"
    )

    respuesta = requests.get(url)

    return respuesta.json()

In [ ]:
datos_ruta_1 = obtener_ruta_carretera(
    ruta_1,
    coordenadas_ruta_1
)

In [ ]:
ruta_osrm = datos_ruta_1["routes"][0]

distancia_km = ruta_osrm["distance"] / 1000

tiempo_horas = ruta_osrm["duration"] / 3600

print("RESULTADOS RUTA 1")
print("===================")

print(
    f"Distancia: {distancia_km:,.2f} km"
)

print(
    f"Tiempo estimado: {tiempo_horas:,.2f} horas"
)

RESULTADOS RUTA 1
Distancia: 2,490.02 km
Tiempo estimado: 32.18 horas


In [ ]:
geometria = ruta_osrm["geometry"]

folium.GeoJson(
    geometria,
    name="Ruta 1 - Carretera"
).add_to(mapa_ruta_1)

folium.LayerControl().add_to(
    mapa_ruta_1
)

mapa_ruta_1

In [ ]:
ruta_2 = [
    "León",
    "Topolobampo",
    "La Paz",
    "Cabo San Lucas",
    "San José del Cabo",


]

print("Ruta 2 - BAJA CALIFORNIA")
print("=" * 30)

for i, ciudad in enumerate(ruta_2, start=1):
  print(f"{i}. {ciudad}")

Ruta 2 - BAJA CALIFORNIA
1. León
2. Topolobampo
3. La Paz
4. Cabo San Lucas
5. San José del Cabo


In [ ]:
coordenadas_ruta_2 = {
    "León": (21.1220, -101.6840),

    "Topolobampo": (25.6008, -109.0522),

    "La Paz": (24.1422, -110.3108),

    "Cabo San Lucas": (22.8897, -109.9156),

    "San José del Cabo": (23.0614, -109.7031)
}

In [ ]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in kilometers

    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    distance = R * c
    return distance

topolobampo_coords = coordenadas_ruta_2["Topolobampo"]
la_paz_coords = coordenadas_ruta_2["La Paz"]

distancia_maritima = haversine(
    topolobampo_coords[0],
    topolobampo_coords[1],
    la_paz_coords[0],
    la_paz_coords[1]
)

print(f"Distancia marítima (línea recta) entre Topolobampo y La Paz: {distancia_maritima:,.2f} km")

# Add the straight line to mapa_ruta_2
folium.PolyLine(
    locations=[
        topolobampo_coords,
        la_paz_coords
    ],
    color="green",
    weight=5,
    tooltip="Ruta Marítima (Directa)"
).add_to(mapa_ruta_2)

mapa_ruta_2

Distancia marítima (línea recta) entre Topolobampo y La Paz: 205.97 km


In [ ]:
# Segmento de carretera: León a Topolobampo
segmento_leon_topolobampo = ["León", "Topolobampo"]
datos_leon_topolobampo = obtener_ruta_carretera(segmento_leon_topolobampo, coordenadas_ruta_2)
geometria_leon_topolobampo = datos_leon_topolobampo["routes"][0]["geometry"]
folium.GeoJson(
    geometria_leon_topolobampo,
    name="León a Topolobampo (Carretera)",
    style_function=lambda x: {"color": "blue", "weight": 5}
).add_to(mapa_ruta_2)

# Segmento de carretera: La Paz a San José del Cabo
segmento_la_paz_sjc = ["La Paz", "San José del Cabo"]
datos_la_paz_sjc = obtener_ruta_carretera(segmento_la_paz_sjc, coordenadas_ruta_2)
geometria_la_paz_sjc = datos_la_paz_sjc["routes"][0]["geometry"]
folium.GeoJson(
    geometria_la_paz_sjc,
    name="La Paz a San José del Cabo (Carretera)",
    style_function=lambda x: {"color": "blue", "weight": 5}
).add_to(mapa_ruta_2)

# Segmento de carretera: San José del Cabo a Cabo San Lucas
segmento_sjc_cabo = ["San José del Cabo", "Cabo San Lucas"]
datos_sjc_cabo = obtener_ruta_carretera(segmento_sjc_cabo, coordenadas_ruta_2)
geometria_sjc_cabo = datos_sjc_cabo["routes"][0]["geometry"]
folium.GeoJson(
    geometria_sjc_cabo,
    name="San José del Cabo a Cabo San Lucas (Carretera)",
    style_function=lambda x: {"color": "blue", "weight": 5}
).add_to(mapa_ruta_2)

folium.LayerControl().add_to(mapa_ruta_2)

mapa_ruta_2

In [ ]:
total_distancia_ruta_2 = 0
total_tiempo_ruta_2 = 0

# Distancia y tiempo León a Topolobampo (carretera)
total_distancia_ruta_2 += datos_leon_topolobampo["routes"][0]["distance"] / 1000
total_tiempo_ruta_2 += datos_leon_topolobampo["routes"][0]["duration"] / 3600

# Distancia marítima Topolobampo a La Paz (línea recta)
total_distancia_ruta_2 += distancia_maritima

# Distancia y tiempo La Paz a San José del Cabo (carretera)
total_distancia_ruta_2 += datos_la_paz_sjc["routes"][0]["distance"] / 1000
total_tiempo_ruta_2 += datos_la_paz_sjc["routes"][0]["duration"] / 3600

# Distancia y tiempo San José del Cabo a Cabo San Lucas (carretera)
total_distancia_ruta_2 += datos_sjc_cabo["routes"][0]["distance"] / 1000
total_tiempo_ruta_2 += datos_sjc_cabo["routes"][0]["duration"] / 3600

print("RESULTADOS RUTA 2 (Combinada)")
print("==========================")
print(f"Distancia total estimada: {total_distancia_ruta_2:,.2f} km")
print(f"Tiempo total estimado (solo carretera): {total_tiempo_ruta_2:,.2f} horas")
print("Nota: El tiempo marítimo no está incluido en el cálculo total de tiempo.")

NameError: name 'datos_leon_topolobampo' is not defined

In [ ]:
ruta_3 = [
    "León",
    "Zacatecas",
    "Durango",
    "Chihuahua",

]

print("Ruta 3 - CENTRO-NORTE")
print("=" * 30)

for i, ciudad in enumerate(ruta_3, start=1):
  print(f"{i}. {ciudad}")

Ruta 3 - CENTRO-NORTE
1. León
2. Zacatecas
3. Durango
4. Chihuahua


In [ ]:
ruta_3 = [
    "León",
    "Zacatecas",
    "Durango",
    "Chihuahua",



]

print("Ruta 3 - CENTRO-NORTE")
print("=" * 30)

for i, ciudad in enumerate(ruta_3, start=1):
  print(f"{i}. {ciudad}")

Ruta 3 - CENTRO-NORTE
1. León
2. Zacatecas
3. Durango
4. Chihuahua


In [ ]:
coordenadas_ruta_3 = {
    "León": (21.1220, -101.6840),

    "Zacatecas": (22.7717, -102.5753),

    "Durango": (24.0277, -104.6532),

    "Chihuahua": (28.6330, -106.0691),


}

In [ ]:
import requests

def obtener_ruta_carretera(ciudades, coordenadas):

    puntos = []

    for ciudad in ciudades:

        latitud, longitud = coordenadas[ciudad]

        puntos.append(
            f"{longitud},{latitud}"
        )

    cadena = ";".join(puntos)

    url = (
        "https://router.project-osrm.org/"
        "route/v1/driving/"
        + cadena +
        "?overview=full"
        "&geometries=geojson"
    )

    respuesta = requests.get(url)

    return respuesta.json()

In [ ]:
import folium

mapa_ruta_3 = folium.Map(
    location=[25, -107],
    zoom_start=5
)

for numero, ciudad in enumerate(ruta_3, start=1):

    latitud, longitud = coordenadas_ruta_3[ciudad]

    color = "red" if ciudad == "León" else "blue"

    folium.Marker(
        location=[latitud, longitud],
        popup=f"{numero}. {ciudad}",
        tooltip=f"{numero}. {ciudad}",
        icon=folium.Icon(color=color)
    ).add_to(mapa_ruta_3)

mapa_ruta_3

In [ ]:
def obtener_ruta(origen, destino):

    lon1 = origen[1]
    lat1 = origen[0]

    lon2 = destino[1]
    lat2 = destino[0]

    url = (
        f"https://router.project-osrm.org/"
        f"route/v1/driving/"
        f"{lon1},{lat1};{lon2},{lat2}"
        f"?overview=false"
    )

    respuesta = requests.get(url)

    datos_ruta = respuesta.json()

    if datos_ruta["code"] == "Ok":

        ruta = datos_ruta["routes"][0]

        distancia = ruta["distance"] / 1000
        tiempo = ruta["duration"] / 60

        return distancia, tiempo

    else:

        return None, None

In [ ]:
datos_ruta_3 = obtener_ruta_carretera(
    ruta_3,
    coordenadas_ruta_3
)

In [ ]:
ruta_osrm = datos_ruta_3["routes"][0]

distancia_km = ruta_osrm["distance"] / 1000

tiempo_horas = ruta_osrm["duration"] / 3600

print("RESULTADOS RUTA 3")
print("===================")

print(
    f"Distancia: {distancia_km:,.2f} km"
)

print(
    f"Tiempo estimado: {tiempo_horas:,.2f} horas"
)

RESULTADOS RUTA 3
Distancia: 1,216.90 km
Tiempo estimado: 15.62 horas


In [ ]:
geometria = ruta_osrm["geometry"]

folium.GeoJson(
    geometria,
    name="Ruta 3 - Carretera"
).add_to(mapa_ruta_3)

folium.LayerControl().add_to(
    mapa_ruta_3
)

mapa_ruta_3

In [ ]:
ruta_4 = [
    "León",
    "Zacatecas",
    "Durango",
    "Torreón",
    "Cuatro Ciénegas",
    "Saltillo",
    "Monterrey",
]

print("Ruta 4 - CENTRO-NORESTE")
print("=" * 30)

for i, ciudad in enumerate(ruta_4, start=1):
  print(f"{i}. {ciudad}")

Ruta 4 - CENTRO-NORESTE
1. León
2. Zacatecas
3. Durango
4. Torreón
5. Cuatro Ciénegas
6. Saltillo
7. Monterrey


In [ ]:
coordenadas_ruta_4 = {
    "León": (21.1220, -101.6840),

    "Zacatecas": (22.7717, -102.5753),

    "Durango": (24.0277, -104.6532),

    "Torreón": (25.5439, -103.4190),

    "Cuatro Ciénegas": (26.9861, -102.0664),

    "Saltillo": (25.4230, -100.9928),

    "Monterrey": (25.6843, -100.3172)


}

In [ ]:
import requests

def obtener_ruta_carretera(ciudades, coordenadas):

    puntos = []

    for ciudad in ciudades:

        latitud, longitud = coordenadas[ciudad]

        puntos.append(
            f"{longitud},{latitud}"
        )

    cadena = ";".join(puntos)

    url = (
        "https://router.project-osrm.org/"
        "route/v1/driving/"
        + cadena +
        "?overview=full"
        "&geometries=geojson"
    )

    respuesta = requests.get(url)

    return respuesta.json()

In [ ]:
import folium

mapa_ruta_4 = folium.Map(
    location=[25, -107],
    zoom_start=5
)

for numero, ciudad in enumerate(ruta_4, start=1):

    latitud, longitud = coordenadas_ruta_4[ciudad]

    color = "red" if ciudad == "León" else "blue"

    folium.Marker(
        location=[latitud, longitud],
        popup=f"{numero}. {ciudad}",
        tooltip=f"{numero}. {ciudad}",
        icon=folium.Icon(color=color)
    ).add_to(mapa_ruta_4)

mapa_ruta_4

In [ ]:
def obtener_ruta(origen, destino):

    lon1 = origen[1]
    lat1 = origen[0]

    lon2 = destino[1]
    lat2 = destino[0]

    url = (
        f"https://router.project-osrm.org/"
        f"route/v1/driving/"
        f"{lon1},{lat1};{lon2},{lat2}"
        f"?overview=false"
    )

    respuesta = requests.get(url)

    datos_ruta = respuesta.json()

    if datos_ruta["code"] == "Ok":

        ruta = datos_ruta["routes"][0]

        distancia = ruta["distance"] / 1000
        tiempo = ruta["duration"] / 60

        return distancia, tiempo

    else:

        return None, None

In [ ]:
datos_ruta_4 = obtener_ruta_carretera(
    ruta_4,
    coordenadas_ruta_4
)

In [ ]:
ruta_osrm = datos_ruta_4["routes"][0]

distancia_km = ruta_osrm["distance"] / 1000

tiempo_horas = ruta_osrm["duration"] / 3600

print("RESULTADOS RUTA 4")
print("===================")

print(
    f"Distancia: {distancia_km:,.2f} km"
)

print(
    f"Tiempo estimado: {tiempo_horas:,.2f} horas"
)

RESULTADOS RUTA 4
Distancia: 1,382.51 km
Tiempo estimado: 19.08 horas


In [ ]:
geometria = ruta_osrm["geometry"]

folium.GeoJson(
    geometria,
    name="Ruta 4 - Carretera"
).add_to(mapa_ruta_4)

folium.LayerControl().add_to(
    mapa_ruta_4
)

mapa_ruta_4

In [ ]:
ruta_5 = [
    "León",
    "San Luis Potosí",
    "Tampico",
    "Reynosa",
]

print("Ruta 5 - NORTE-GOLFO")
print("=" * 30)

for i, ciudad in enumerate(ruta_5, start=1):
  print(f"{i}. {ciudad}")

Ruta 5 - NORTE-GOLFO
1. León
2. San Luis Potosí
3. Tampico
4. Reynosa


In [ ]:
coordenadas_ruta_5 = {
    "León": (21.1220, -101.6840),

    "San Luis Potosí": (22.1516, -100.9764),

    "Tampico": (22.2372, -97.8700),

    "Reynosa": (26.0704, -98.2964),


}

In [ ]:
import requests

def obtener_ruta_carretera(ciudades, coordenadas):

    puntos = []

    for ciudad in ciudades:

        latitud, longitud = coordenadas[ciudad]

        puntos.append(
            f"{longitud},{latitud}"
        )

    cadena = ";".join(puntos)

    url = (
        "https://router.project-osrm.org/"
        "route/v1/driving/"
        + cadena +
        "?overview=full"
        "&geometries=geojson"
    )

    respuesta = requests.get(url)

    return respuesta.json()

In [ ]:
import folium

mapa_ruta_5 = folium.Map(
    location=[25, -107],
    zoom_start=5
)

for numero, ciudad in enumerate(ruta_5, start=1):

    latitud, longitud = coordenadas_ruta_5[ciudad]

    color = "red" if ciudad == "León" else "blue"

    folium.Marker(
        location=[latitud, longitud],
        popup=f"{numero}. {ciudad}",
        tooltip=f"{numero}. {ciudad}",
        icon=folium.Icon(color=color)
    ).add_to(mapa_ruta_5)

mapa_ruta_5

In [ ]:
def obtener_ruta(origen, destino):

    lon1 = origen[1]
    lat1 = origen[0]

    lon2 = destino[1]
    lat2 = destino[0]

    url = (
        f"https://router.project-osrm.org/"
        f"route/v1/driving/"
        f"{lon1},{lat1};{lon2},{lat2}"
        f"?overview=false"
    )

    respuesta = requests.get(url)

    datos_ruta = respuesta.json()

    if datos_ruta["code"] == "Ok":

        ruta = datos_ruta["routes"][0]

        distancia = ruta["distance"] / 1000
        tiempo = ruta["duration"] / 60

        return distancia, tiempo

    else:

        return None, None

In [ ]:
datos_ruta_5 = obtener_ruta_carretera(
    ruta_5,
    coordenadas_ruta_5
)

In [ ]:
ruta_osrm = datos_ruta_5["routes"][0]

distancia_km = ruta_osrm["distance"] / 1000

tiempo_horas = ruta_osrm["duration"] / 3600

print("RESULTADOS RUTA 5")
print("===================")

print(
    f"Distancia: {distancia_km:,.2f} km"
)

print(
    f"Tiempo estimado: {tiempo_horas:,.2f} horas"
)

RESULTADOS RUTA 5
Distancia: 1,057.27 km
Tiempo estimado: 15.88 horas


In [ ]:
geometria = ruta_osrm["geometry"]

folium.GeoJson(
    geometria,
    name="Ruta 5 - Carretera"
).add_to(mapa_ruta_5)

folium.LayerControl().add_to(
    mapa_ruta_5
)

mapa_ruta_5